# 04 Live Dissolved Oxygen Prediction

This notebook demonstrates one live dissolved oxygen prediction run.

It uses:
- Latest AquaSensor observations
- Live weather from Open-Meteo
- Previously trained machine learning models

The output is saved to:

`data/processed/live_river_do_forecasts.csv`


Cell 1 — Markdown
Cell 2 — Imports
Cell 3 — File Paths and Settings
Cell 4 — Helper Function for Model File Names
Cell 5 — Download Live Weather from Open-Meteo
Cell 6 — Create Live Time Features
Cell 7 — Load Latest AquaSensor Readings
Cell 8 — Load Best Trained Models
Cell 9 — Live Prediction Function
Cell 10 — Run One Live Prediction
Cell 11 — Display Saved Live Prediction File
Cell 12 - Summary

In [1]:
import os
import time
import joblib
import requests
import pandas as pd

from datetime import datetime

In [2]:
AQUASENSOR_FINAL = "../data/processed/aquasensor_final.csv"
PERFORMANCE_FILE = "../data/processed/do_prediction_model_performance.csv"
OUTPUT_FILE = "../data/processed/live_river_do_forecasts.csv"
MODELS_DIR = "../models"

LATITUDE = 53.33
LONGITUDE = -1.65

HORIZONS = [
    "15min", "30min", "45min", "60min",
    "75min", "90min", "105min", "120min"
]

FEATURES = [
    "temperature",
    "air_temperature_c",
    "sunshine_wm2",
    "hour",
    "dissolved_oxygen_mgl",
    "dissolved_oxygen_pct",
    "pollution_alert",
    "anomaly_type",
    "season_proxy",
    "sensor_id",
    "sensor_name",
]

In [3]:
def safe_filename_text(text):
    return (
        text.lower()
        .replace(" ", "_")
        .replace("%", "pct")
        .replace("/", "_per_")
    )

In [4]:
def get_live_weather():
    """
    Live weather source:
    Open-Meteo Forecast API
    https://api.open-meteo.com
    """

    url = (
        "https://api.open-meteo.com/v1/forecast"
        f"?latitude={LATITUDE}"
        f"&longitude={LONGITUDE}"
        "&hourly=temperature_2m,shortwave_radiation,cloud_cover"
        "&forecast_days=1"
        "&timezone=Europe/London"
    )

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    data = response.json()
    hourly = data["hourly"]

    weather = pd.DataFrame(
        {
            "timestamp": pd.to_datetime(hourly["time"]),
            "air_temperature_c": hourly["temperature_2m"],
            "sunshine_wm2": hourly["shortwave_radiation"],
            "cloud_cover_pct": hourly["cloud_cover"],
        }
    )

    now = pd.Timestamp.now()
    weather["time_diff"] = (weather["timestamp"] - now).abs()

    latest_weather = weather.sort_values("time_diff").iloc[0]

    return {
        "air_temperature_c": latest_weather["air_temperature_c"],
        "sunshine_wm2": latest_weather["sunshine_wm2"],
        "cloud_cover_pct": latest_weather["cloud_cover_pct"],
    }

In [5]:
def add_live_time_features(row):
    timestamp = pd.to_datetime(row["timestamp"])

    row["hour"] = timestamp.hour
    row["month"] = timestamp.month
    row["day_of_year"] = timestamp.dayofyear

    season_map = {
        12: 0, 1: 0, 2: 0,
        3: 1, 4: 1, 5: 1,
        6: 2, 7: 2, 8: 2,
        9: 3, 10: 3, 11: 3,
    }

    row["season"] = season_map[row["month"]]

    angle = 2 * 3.141592653589793 * (row["day_of_year"] / 365.25)
    seasonal_signal = (
        pd.Series([angle])
        .apply(lambda x: __import__("math").cos(x - 3.141592653589793))
        .iloc[0] + 1
    ) / 2

    clear_sky_signal = 1 - (row["cloud_cover_pct"] / 100)

    row["season_proxy"] = round(
        0.70 * seasonal_signal + 0.30 * clear_sky_signal,
        4
    )

    return row

In [6]:
def load_latest_sensor_rows():
    df = pd.read_csv(AQUASENSOR_FINAL, parse_dates=["timestamp"], low_memory=False)

    df["sensor_id"] = df["sensor_id"].astype(str)
    df["sensor_name"] = df["sensor_name"].astype(str)

    latest_rows = (
        df.sort_values("timestamp")
        .groupby("sensor_id")
        .tail(1)
        .reset_index(drop=True)
    )

    return latest_rows

In [7]:
def get_best_model_name(target_type, horizon):
    metrics = pd.read_csv(PERFORMANCE_FILE)

    selected = metrics[
        (metrics["target_type"] == target_type) &
        (metrics["horizon"] == horizon)
    ]

    best_row = selected.sort_values("RMSE").iloc[0]
    return best_row["model"]


def load_model(model_name, target_type, horizon):
    model_file = (
        f"{safe_filename_text(model_name)}_"
        f"{safe_filename_text(target_type)}_"
        f"{horizon}.pkl"
    )

    model_path = os.path.join(MODELS_DIR, model_file)

    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model not found: {model_path}")

    return joblib.load(model_path)

In [8]:
def make_live_prediction():
    live_weather = get_live_weather()
    latest_rows = load_latest_sensor_rows()

    prediction_run_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    forecast_rows = []

    print("\n" + "=" * 80)
    print("LIVE RIVER DO FORECAST")
    print("=" * 80)
    print(f"Prediction run time: {prediction_run_time}")
    print("Weather source: Open-Meteo Forecast API")
    print(f"Weather location: lat={LATITUDE}, lon={LONGITUDE}")
    print(f"Latest sensor rows used: {len(latest_rows)}")

    for _, row in latest_rows.iterrows():
        row = row.copy()

        row["air_temperature_c"] = live_weather["air_temperature_c"]
        row["sunshine_wm2"] = live_weather["sunshine_wm2"]
        row["cloud_cover_pct"] = live_weather["cloud_cover_pct"]

        row = add_live_time_features(row)

        input_row = pd.DataFrame([row[FEATURES]])

        result = {
            "prediction_run_time": prediction_run_time,
            "latest_sensor_timestamp": row["timestamp"],
            "sensor_id": row["sensor_id"],
            "sensor_name": row["sensor_name"],
            "current_do_mgl": row["dissolved_oxygen_mgl"],
            "current_do_pct": row["dissolved_oxygen_pct"],
            "air_temperature_c": row["air_temperature_c"],
            "sunshine_wm2": row["sunshine_wm2"],
            "cloud_cover_pct": row["cloud_cover_pct"],
            "pollution_alert": row["pollution_alert"],
            "anomaly_type": row["anomaly_type"],
        }

        for horizon in HORIZONS:
            mgl_model_name = get_best_model_name("DO mg/L", horizon)
            pct_model_name = get_best_model_name("DO %", horizon)

            mgl_model = load_model(mgl_model_name, "DO mg/L", horizon)
            pct_model = load_model(pct_model_name, "DO %", horizon)

            result[f"predicted_do_mgl_{horizon}"] = mgl_model.predict(input_row)[0]
            result[f"predicted_do_pct_{horizon}"] = pct_model.predict(input_row)[0]

        forecast_rows.append(result)

    forecast_df = pd.DataFrame(forecast_rows)

    if os.path.exists(OUTPUT_FILE):
        old_df = pd.read_csv(OUTPUT_FILE)
        final_df = pd.concat([old_df, forecast_df], ignore_index=True)
    else:
        final_df = forecast_df

    final_df.to_csv(OUTPUT_FILE, index=False)

    display_cols = [
        "sensor_name",
        "current_do_mgl",
        "current_do_pct",
        "predicted_do_mgl_15min",
        "predicted_do_mgl_30min",
        "predicted_do_mgl_45min",
        "predicted_do_mgl_60min",
        "predicted_do_mgl_120min",
    ]

    print("\nLive predictions:")
    print(forecast_df[display_cols].to_string(index=False))

    print(f"\nSaved live forecasts to: {OUTPUT_FILE}")

    return forecast_df

In [9]:
live_predictions = make_live_prediction()
live_predictions


LIVE RIVER DO FORECAST
Prediction run time: 2026-07-07 16:35:49
Weather source: Open-Meteo Forecast API
Weather location: lat=53.33, lon=-1.65
Latest sensor rows used: 3

Live predictions:
  sensor_name  current_do_mgl  current_do_pct  predicted_do_mgl_15min  predicted_do_mgl_30min  predicted_do_mgl_45min  predicted_do_mgl_60min  predicted_do_mgl_120min
   Derwent 21            11.2            95.8               11.217626               11.232944               11.250148               11.270311                11.339697
Derwent 13-50             9.9            92.1                9.902995                9.907503                9.914010                9.922278                 9.949579
   Derwent 13             5.5            51.5                5.547659                5.610111                5.681081                5.757415                 6.067476

Saved live forecasts to: ../data/processed/live_river_do_forecasts.csv


,prediction_run_time,latest_sensor_timestamp,sensor_id,sensor_name,current_do_mgl,current_do_pct,air_temperature_c,sunshine_wm2,cloud_cover_pct,pollution_alert,...,predicted_do_mgl_60min,predicted_do_pct_60min,predicted_do_mgl_75min,predicted_do_pct_75min,predicted_do_mgl_90min,predicted_do_pct_90min,predicted_do_mgl_105min,predicted_do_pct_105min,predicted_do_mgl_120min,predicted_do_pct_120min
0,2026-07-07 16:35:49,2026-06-01 05:30:34,941205,Derwent 21,11.2,95.8,20.5,420.0,70,0,...,11.270311,96.856229,11.288012,97.118062,11.305997,97.378287,11.322432,97.630371,11.339697,97.884818
1,2026-07-07 16:35:49,2026-06-04 21:09:04,941115,Derwent 13-50,9.9,92.1,20.5,420.0,70,0,...,9.922278,92.426004,9.929291,92.506461,9.936314,92.584342,9.942610,92.657975,9.949579,92.734487
2,2026-07-07 16:35:49,2026-06-04 21:13:40,sensor022,Derwent 13,5.5,51.5,20.5,420.0,70,0,...,5.757415,53.890614,5.832343,54.551027,5.909022,55.235948,5.986460,55.941190,6.067476,56.685890


In [10]:
saved_live_predictions = pd.read_csv(OUTPUT_FILE)
saved_live_predictions.tail()

,prediction_run_time,latest_sensor_timestamp,sensor_id,sensor_name,current_do_mgl,current_do_pct,pollution_alert,anomaly_type,predicted_do_mgl_15min,predicted_do_pct_15min,...,predicted_do_pct_75min,predicted_do_mgl_90min,predicted_do_pct_90min,predicted_do_mgl_105min,predicted_do_pct_105min,predicted_do_mgl_120min,predicted_do_pct_120min,air_temperature_c,sunshine_wm2,cloud_cover_pct
142,2026-06-29 15:32:35,2026-06-04 21:09:04,941115,Derwent 13-50,9.9,92.1,0,0,9.904924,92.193378,...,92.569194,9.941272,92.652017,9.947786,92.728665,9.954785,92.805130,19.5,434.0,100.0
143,2026-06-29 15:32:35,2026-06-04 21:13:40,sensor022,Derwent 13,5.5,51.5,0,0,5.549587,52.063612,...,54.613760,5.913980,55.303623,5.991636,56.011879,6.072682,56.756533,19.5,434.0,100.0
144,2026-07-07 16:35:49,2026-06-01 05:30:34,941205,Derwent 21,11.2,95.8,0,0,11.217626,96.038545,...,97.118062,11.305997,97.378287,11.322432,97.630371,11.339697,97.884818,20.5,420.0,70.0
145,2026-07-07 16:35:49,2026-06-04 21:09:04,941115,Derwent 13-50,9.9,92.1,0,0,9.902995,92.172918,...,92.506461,9.936314,92.584342,9.942610,92.657975,9.949579,92.734487,20.5,420.0,70.0
146,2026-07-07 16:35:49,2026-06-04 21:13:40,sensor022,Derwent 13,5.5,51.5,0,0,5.547659,52.043151,...,54.551027,5.909022,55.235948,5.986460,55.941190,6.067476,56.685890,20.5,420.0,70.0


## Summary

This notebook demonstrates one live dissolved oxygen prediction run.

The live prediction system uses the latest AquaSensor readings together with live weather data from the Open-Meteo Forecast API.

The trained models predict dissolved oxygen in both mg/L and percentage for 15, 30, 45, 60, 75, 90, 105, and 120 minutes ahead.

The results are saved to:

`data/processed/live_river_do_forecasts.csv`